In [16]:
import requests

url = "https://apis.data.go.kr/B410001/trend-news"

params = {
    "serviceKey": "V/WBD6uRvTL7UmWfZq1iXcdUJSrzipq3qPcddtskwBuwLhZkHOXp7cLL6i4C41xqLbH7jtCT8QJ5MJCn3Cw32Q==",  # 공공데이터포털의 일반 인증키(Decoding)
    "type": "json",
    "numOfRows": 100,
    "pageNo": 1,
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, params=params, headers=headers, timeout=20)

print("status:", res.status_code)
print("final url:", res.url)
print("content-type:", res.headers.get("content-type"))
print(res.text[:1000])

status: 500
final url: https://apis.data.go.kr/B410001/trend-news?serviceKey=V%2FWBD6uRvTL7UmWfZq1iXcdUJSrzipq3qPcddtskwBuwLhZkHOXp7cLL6i4C41xqLbH7jtCT8QJ5MJCn3Cw32Q%3D%3D&type=json&numOfRows=100&pageNo=1
content-type: text/plain; charset=utf-8
Unexpected errors



# 전체 뉴스 저장

In [ ]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

SERVICE_KEY = os.getenv("SERVICE_KEY_DECODED")
url = "https://apis.data.go.kr/B410001/kotra_overseasMarketNews/ovseaMrktNews/ovseaMrktNews"

params = {
    "serviceKey": SERVICE_KEY,
    "type": "json",
    "numOfRows": 1000,
    "pageNo": 1,

    # # 선택 검색 조건
    # "search1": "일본",        # 국가명
    # "search2": "뷰티",        # 뉴스 제목
    # "search4": "20250501",   # 시작일
    # "search5": "",           # 산업분류 코드
    # "search6": "",           # Hot Clip
    # "search7": "20260504",   # 종료일
}

res = requests.get(url, params=params, timeout=10)
data = res.json()

print("status:", res.status_code)
print("url:", res.url)

In [ ]:
body = data["response"]["body"]
items = body["itemList"]["item"]

df = pd.DataFrame(items)

# 컬럼명 한글화
rename_cols = {
    "newsTitl": "뉴스제목",
    "kotraNewsUrl": "게시물 URL",
    "fileLink": "파일링크",
    "othbcDt": "뉴스게시일자",
    "natn": "국가",
    "regn": "지역",
    "ovrofInfo": "무역관정보",
    "newsWrterNm": "뉴스작성자명",
    "infoCl": "정보분류(메뉴명)",
    "indstCl": "산업분류",
    "indstCdList": "산업분류코드목록",
    "hsCdNm": "HS코드명",
    "cmdltNmKorn": "품목명한글",
    "cmdltNmEng": "품목명영문",
    "dataType": "저작권 정책",
    "bbstxSn": "게시글일련번호",
    "bbsSn": "정보분류(메뉴ID)",
    "jobSeNm": "일자리구분명",
}

df = df.rename(columns=rename_cols)
print("전체 건수:", body["totalCnt"])
df

In [ ]:
import requests
import pandas as pd
import os
import math
import time
import sqlite3
from dotenv import load_dotenv

load_dotenv()

SERVICE_KEY = os.getenv("SERVICE_KEY_DECODED")

url = "https://apis.data.go.kr/B410001/kotra_overseasMarketNews/ovseaMrktNews/ovseaMrktNews"

num_rows = 100

# 1페이지 호출해서 전체 건수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "type": "json",
    "numOfRows": num_rows,
    "pageNo": 1,
}

rename_cols = {
    "newsTitl": "뉴스제목",
    "kotraNewsUrl": "게시물 URL",
    "fileLink": "파일링크",
    "othbcDt": "뉴스게시일자",
    "natn": "국가",
    "regn": "지역",
    "ovrofInfo": "무역관정보",
    "newsWrterNm": "뉴스작성자명",
    "infoCl": "정보분류(메뉴명)",
    "indstCl": "산업분류",
    "indstCdList": "산업분류코드목록",
    "hsCdNm": "HS코드명",
    "cmdltNmKorn": "품목명한글",
    "cmdltNmEng": "품목명영문",
    "dataType": "저작권 정책",
    "bbstxSn": "게시글일련번호",
    "bbsSn": "정보분류(메뉴ID)",
    "jobSeNm": "일자리구분명",
}

res = requests.get(url, params=params, timeout=30)
data = res.json()

body = data["response"]["body"]
total_cnt = int(body["totalCnt"])
total_pages = math.ceil(total_cnt / num_rows)

print("전체 건수:", total_cnt)
print("전체 페이지:", total_pages)


all_items = []

for page in range(1, total_pages + 1):
    params = {
        "serviceKey": SERVICE_KEY,
        "type": "json",
        "numOfRows": num_rows,
        "pageNo": page,
    }

    success = False

    for attempt in range(3):
        try:
            res = requests.get(url, params=params, timeout=30)
            data = res.json()

            items = data["response"]["body"]["itemList"]["item"]

            if isinstance(items, dict):
                items = [items]

            all_items.extend(items)

            print(f"{page}/{total_pages} 페이지 완료")
            success = True
            break

        except Exception as e:
            print(f"{page}페이지 오류, {attempt + 1}번째 재시도:", e)
            time.sleep(2)

    if not success:
        print(f"{page}페이지는 최종 실패. 다음 페이지로 넘어갑니다.")

    time.sleep(0.3)

df = pd.DataFrame(all_items)
df = df.rename(columns=rename_cols)

# SQLite 저장
conn = sqlite3.connect("kotra_news.db")
df.to_sql("kotra_news", conn, if_exists="replace", index=False)
conn.close()

# Excel 저장
df.to_excel("kotra_news.xlsx", index=False)

print("저장 완료")
print("수집 행 수:", len(df))

# 뉴스 분석

In [7]:
import pandas as pd

df = pd.read_excel("data/kotra_news.xlsx")

cols = ["지역", "국가", "산업분류", "정보분류(메뉴명)"]

for col in cols:
    print("=" * 80)
    print(f"[{col}]")
    print("전체 행 수:", len(df))
    print("결측치 개수:", df[col].isna().sum())
    print("결측치 비율:", round(df[col].isna().mean() * 100, 2), "%")
    print("unique 개수:", df[col].nunique(dropna=True))

    print("\n값별 개수:")
    print(df[col].value_counts(dropna=False))

[지역]
전체 행 수: 94572
결측치 개수: 8
결측치 비율: 0.01 %
unique 개수: 9

값별 개수:
지역
아시아         38650
유럽          22230
북미          11507
중남미          8753
중동           8014
아프리카         2898
대양주          2121
KOTRA 본사      389
NaN             8
전세계             2
Name: count, dtype: int64
[국가]
전체 행 수: 94572
결측치 개수: 770
결측치 비율: 0.81 %
unique 개수: 114

값별 개수:
국가
중국       12144
미국        9549
일본        5906
러시아연방     3449
인도        3225
         ...  
바레인          1
예멘           1
엘살바도르        1
아이티          1
알바니아         1
Name: count, Length: 115, dtype: int64
[산업분류]
전체 행 수: 94572
결측치 개수: 26754
결측치 비율: 28.29 %
unique 개수: 2187

값별 개수:
산업분류
NaN                                                     26754
전자/전기                                                   16228
기타                                                       7254
산업일반                                                     5074
자동차/수송기기                                                 4781
                                                        ... 

In [ ]:
import pandas as pd

df = pd.read_excel("kotra_news.xlsx")
df["뉴스게시일자"] = pd.to_datetime(df["뉴스게시일자"], errors="coerce")
df["연도"] = df["뉴스게시일자"].dt.year

# 연도별 뉴스 개수
year_counts = (
    df["연도"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)

year_counts.columns = ["연도", "뉴스개수"]
year_counts

# 기사 본문 내용 저장

In [ ]:
import pandas as pd

df = pd.read_excel("data/kotra_news.xlsx")
df.info()

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

df = pd.read_excel("kotra_news.xlsx")
first_url = df.loc[0, "게시물 URL"] # 첫 번째 행의 게시물 URL

print(first_url)

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(first_url, headers=headers, timeout=20)

print("status:", res.status_code)
print("url:", res.url)

In [ ]:
soup = BeautifulSoup(res.text, "html.parser")
keyword_area = soup.select_one("div.keywordArea")

if keyword_area:
    keywords = [a.get_text(strip=True) for a in keyword_area.select("a")]
else:
    keywords = []

print(keywords)

In [ ]:
view_txt = soup.select_one("div.view_txt")

if view_txt:
    body_text = view_txt.get_text(separator="\n", strip=True)
else:
    body_text = ""

print(body_text[:200])

In [ ]:
view_data = soup.select_one("div.view_txt div.viewDataWrap")

if view_data:
    body_text = view_data.get_text(separator="\n", strip=True)
else:
    body_text = ""

print(body_text[:200])

In [ ]:
result = {
    "게시물 URL": first_url,
    "키워드": keywords,
    "본문": body_text,
}

print("URL:", result["게시물 URL"])
print("키워드:", result["키워드"])
print("본문 앞부분:")
print(result["본문"][:1000])

## 전체 기사 본문 수집 파이프라인

In [ ]:
import re
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

INPUT_XLSX = Path("data/kotra_news.xlsx")
PROGRESS_CSV = Path("data/kotra_news_body_progress.csv")
OUTPUT_CSV = Path("data/kotra_news_body.csv")

SAVE_EVERY = 1000
REQUEST_TIMEOUT = 20
URL_COL = "게시물 URL"
CSV_ENCODING = "utf-8-sig"

headers = {"User-Agent": "Mozilla/5.0"}
unsafe_csv_chars = re.compile(r"[\x00]")


def clean_text(value):
    if value is None:
        return ""
    return unsafe_csv_chars.sub("", str(value))


def save_csv(df, path):
    temp_path = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_csv(temp_path, index=False, encoding=CSV_ENCODING)
    temp_path.replace(path)


def fetch_article(url):
    res = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    res.raise_for_status()

    soup = BeautifulSoup(res.text, "html.parser")

    keyword_area = soup.select_one("div.keywordArea")
    keywords = []
    if keyword_area:
        keywords = [a.get_text(strip=True) for a in keyword_area.select("a")]

    view_data = soup.select_one("div.view_txt div.viewDataWrap")
    view_txt = view_data or soup.select_one("div.view_txt")
    body_text = view_txt.get_text(separator="\n", strip=True) if view_txt else ""

    return clean_text(", ".join(keywords)), clean_text(body_text)


if PROGRESS_CSV.exists():
    df = pd.read_csv(PROGRESS_CSV, encoding=CSV_ENCODING, dtype=str, keep_default_na=False)
    print(f"이어하기: {PROGRESS_CSV}")
else:
    df = pd.read_excel(INPUT_XLSX, dtype=str).fillna("")
    print(f"새로 시작: {INPUT_XLSX}")

for col in ["키워드", "본문"]:
    if col not in df.columns:
        df[col] = ""

pending_mask = df["본문"].isna() | (df["본문"].astype(str).str.strip() == "")
pending_indexes = df.index[pending_mask].tolist()

success_count = 0
fail_count = 0
processed_since_save = 0
saved_batches = 0

pbar = tqdm(pending_indexes, desc="기사 본문 수집")

try:
    for idx in pbar:
        url = df.at[idx, URL_COL] if URL_COL in df.columns else ""

        try:
            if pd.isna(url) or not str(url).strip():
                raise ValueError("URL 없음")

            keywords, body_text = fetch_article(str(url).strip())
            if not body_text.strip():
                raise ValueError("본문 없음")

            df.at[idx, "키워드"] = keywords
            df.at[idx, "본문"] = body_text
            success_count += 1
            last_status = "success"

        except Exception as e:
            fail_count += 1
            last_status = f"fail: {type(e).__name__}"

        processed_since_save += 1
        pbar.set_postfix(success=success_count, fail=fail_count, saved=saved_batches, last=last_status)

        if processed_since_save >= SAVE_EVERY:
            save_csv(df, PROGRESS_CSV)
            saved_batches += 1
            processed_since_save = 0
            pbar.write(f"{success_count + fail_count}개 처리 후 저장 완료: {PROGRESS_CSV}")

except KeyboardInterrupt:
    print(f"중단됨. 다음 실행 시 마지막 저장 파일에서 이어합니다: {PROGRESS_CSV}")

else:
    save_csv(df, PROGRESS_CSV)
    save_csv(df, OUTPUT_CSV)
    print(f"완료: {OUTPUT_CSV}")

print(f"이어하기 파일: {PROGRESS_CSV}")

# OpenAI 임베딩 -> 성능 구림

In [ ]:
import pandas as pd

df = pd.read_csv("data/kotra_news_body.csv")
df.count()

/var/folders/wx/rj6wt5wn1yl8195jd9gqn5jc0000gn/T/ipykernel_58605/3072361545.py:3: DtypeWarning: Columns (0: 일자리구분명) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/kotra_news_body.csv")


품목명한글         23315
뉴스제목          94572
게시물 URL       94572
일자리구분명          908
HS코드명         23343
저작권 정책        94572
품목명영문         23315
지역            94564
파일링크          90405
산업분류          67818
산업분류코드목록      67818
무역관정보         93769
뉴스게시일자        94572
게시글일련번호       94572
뉴스작성자명        94572
정보분류(메뉴ID)    94572
정보분류(메뉴명)     94572
국가            93802
키워드           49125
본문            94550
dtype: int64

In [16]:
df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

df = df[
    df["본문"].notna() &
    (df["본문"].astype(str).str.strip() != "")
].reset_index(drop=True)

# 일단 테스트용으로 첫 10개만
df_sample = df.iloc[:10]
df_sample

/var/folders/wx/rj6wt5wn1yl8195jd9gqn5jc0000gn/T/ipykernel_58605/856752178.py:1: DtypeWarning: Columns (0: 일자리구분명) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")


,품목명한글,뉴스제목,게시물 URL,일자리구분명,HS코드명,저작권 정책,품목명영문,지역,파일링크,산업분류,산업분류코드목록,무역관정보,뉴스게시일자,게시글일련번호,뉴스작성자명,정보분류(메뉴ID),정보분류(메뉴명),국가,키워드,본문
0,"미용이나 메이크업용 제품류와 기초화장용 제품류[의약품은 제외하며, 선스크린(suns...",고물가와 경쟁 속 변화하는 러시아 뷰티 시장,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,3304,4유형,Beauty or make-up preparations and preparation...,유럽,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,노보시비르스크무역관,2026-05-04,240938,김하민,243,트렌드,러시아연방,"#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출",‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모...
1,NaN,"도미니카공화국, 플라스틱 제품류 수입, 유통 기업 대상 환경 규제 강화",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,중남미,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"화학,환경","I001212,I001203",산토도밍고무역관,2026-05-04,241022,Adriana Jimenez,242,경제·무역,도미니카공화국,"#도미니카공화국, #수입 규제, #고형 폐기물법, #환경 규제 준수, #플라스틱 규제",요약\n도미니카공화국은 2025년 12월 법률 제98-25호 시행을 통해 플라스틱·...
2,NaN,HORECA EXPO에서 살펴보는 러시아 외식산업 트렌드와 유망 분야,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,유럽,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"농림수산식품,섬유/패션,생활소비재,유통/물류,관광/교육/서비스,산업일반","I001193,I001197,I001198,I001208,I001210,I001307",모스크바무역관,2026-05-04,240931,정의한,245,현장·인터뷰,러시아연방,"#러시아, #식품, #외식, #식당, #호텔, #레스토랑, #Horeca, #전시회...",<HORECA Expo 2026\n전시회\n개요\n>\n전시회명\nHORECA EX...
3,"미용이나 메이크업용 제품류와 기초화장용 제품류[의약품은 제외하며, 선스크린(suns...",[해외시장뉴스플러스] &lsquo;K-Lifestyle in Chennai&rsqu...,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,3304,4유형,Beauty or make-up preparations and preparation...,아시아,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,첸나이무역관,2026-05-04,240981,윤병은,243,트렌드,인도,"#뷰티, #화장품, #인도, #전자상거래, #온라인, #K-뷰티, #인플루언서, #인터뷰",인도\nK-\n뷰티 시장 확대\n인도 화장품 시장이 빠르게 커지고 있다. 시장조사 ...
4,NaN,2026 케냐 국제 투자 컨퍼런스를 통해 살펴본 케냐 투자유치 동향 및 전망,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,아프리카,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,산업일반,I001307,나이로비무역관,2026-05-04,241000,신정렬,322,투자진출,케냐,"#케냐, #투자, #컨퍼런스","최근 케냐는 '제조업 기반의 경제 성장'이라는 확고한 목표 아래, 파격적인 투자 환..."
5,기타,탄자니아 뷰티시장 성장세&hellip; K-뷰티 진출 기회 커진다,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,330499,4유형,Other\n,아프리카,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,다레살람무역관,2026-05-04,240920,이정훈,243,트렌드,탄자니아,"#K-뷰티, #스킨케어, #화장품시장, #탄자니아, #위조품","탄자니아 뷰티시장 개황\nStatista에 따르면, 탄자니아 코스메틱 시장은 202..."
6,NaN,방범방재종합전 2026 참관기,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,아시아,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,기타,I001211,오사카무역관,2026-05-04,240926,하마다유지,245,현장·인터뷰,일본,"#방재, #지진, #방범, #보안, #열사병",전시회 스케치\n<\n전시회 개요\n>\n전시회\n명\n개 최 장 소\nINTEX ...
7,"기초화장용 제품류,메이크업용 제품류",일본 남성 뷰티 시장 : 전 연령대로 확대,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,"3304991000,3304992000",4유형,"Skin care cosmetics\n,Make-up cosmetics\n",아시아,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,오사카무역관,2026-05-04,240772,김대수,243,트렌드,일본,"#남성, #화장품, #뷰티, #멘즈뷰티, #K-Beauty, #일본, #일본 시장","일본 남성 뷰티 시장, 2019년 278억 엔에서 2024년 497억 엔으로 1.8..."
8,NaN,"인력난 해결의 열쇠, 자동화 기술이 주도하는 일본 식품 산업의 변화",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,아시아,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"농림수산식품,기계류","I001193,I001199",후쿠오카무역관,2026-05-04,240791,김경미,243,트렌드,일본,"#푸드테크, #DX, #AI, #식품, #자동화, #로봇",일본\n식품 가치사슬 전반으로 확산되는 생산성 혁신 흐름\n현재 일본이 정의하는 푸...
9,파이프ㆍ보일러 동체ㆍ탱크ㆍ통이나 이와 유사한 물품에 사용하는 탭ㆍ코크ㆍ밸브와 이와 ...,"美 해양 기자재 시장, &#39;납품 후 끝&#39;은 옛말&hellip; 운영&m...",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,"8481,7307",4유형,Tapsㆍ cocksㆍ valves and similar appliances for...,북미,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"광물/에너지,건설/인프라/플랜트","I001195,I001204",달라스무역관,2026-05-04,241034,신지혜,243,트렌드,미국,"#offshore, #해양에너지, #기자재, #미국, #해양, #장비, #공급, #...","미국 해양에너지 산업은 저유가 기조와 글로벌 공급 과잉이 지속되는 가운데, 트럼프 ..."


In [24]:
from langchain_core.documents import Document

# 3. Document 만들기
docs = []

for idx, row in df_sample.iterrows():
    doc = Document(
        page_content=row["본문"],
        metadata={
            "row_id": idx,
            "url": row.get("게시물 URL", ""),
            "keywords": row.get("키워드", "")
        }
    )
    docs.append(doc)

docs[0:5]

[Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938', 'keywords': '#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출'}, page_content='‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모는\n1\n조2000억\n루블\n(\n약\n144\n억 달러\n)\n로 전년 대비\n7%-8%\n증가했다\n. 반면\n판매되는 화장품\xa0수량은 감소세로\n, ‘Nilson’\n에 따르면 스킨케어 화장품 판매량은\n1\n년 만에\n13.3%, ‘\n헤어케어\n’\n는\n5.3%, ‘\n바디케어\n’\n는\n4.1%\n줄었다\n.\n시장 규모가 커졌음에도 판매량이 감소한\xa0주요 원인으로\xa0화장품\xa0가격의\xa0증가가 꼽힌다.\n<연도별 러시아 화장품 판매수량\n및 증가율>\n(단위: 십\n억 개\n)\n[자료\n: Businesstat.ru]\n오프라인 판매는 감소하는 반면\xa0온라인\xa0마켓플레이스에서의\xa0판매가\xa0증가하고 있다\n. ’25\n년 온라인\xa0소매기업\n‘Wildberries’\n의 스킨케어 화장품 매출은\xa0전년 대비\n28%,\n색조 화장품 매출은\n20%\n증가했다\n.\n온라인 소매점들은 프리미엄\n,\n친환경\n,\n아시아 브랜드 등 다양한 카테고리의 메뉴를 운영하며 소비자들을 유입시키고 있다\n.\n제조\n·\n판매사 구분 모호\n,\n일반 소매점과\xa0뷰티 전문점간\xa0협력 확대\n물류\n‧\n포장 등 비용 상승은 기업이 제조와 판매를 함께\xa0통제하려는 수직통합 압력으로 작동했다\n.\n소비자의\xa0구매력\xa0약화는\xa0멀티기능성\n,\n가성비\n,\n현지\xa0브랜드\xa0선호를\xa0키웠고\n,\n이는\xa0제조와\xa0판매\xa0모두에서\xa0더\xa0촘촘한\xa0가치사

In [26]:
from langchain_text_splitters import CharacterTextSplitter

# 4. Chunking
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
    length_function=len
)

split_docs = splitter.split_documents(docs)
split_docs[0:30]

Created a chunk of size 637, which is longer than the specified 600


[Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938', 'keywords': '#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출'}, page_content='‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모는\n1\n조2000억\n루블\n(\n약\n144\n억 달러\n)\n로 전년 대비\n7%-8%\n증가했다\n. 반면\n판매되는 화장품\xa0수량은 감소세로\n, ‘Nilson’\n에 따르면 스킨케어 화장품 판매량은\n1\n년 만에\n13.3%, ‘\n헤어케어\n’\n는\n5.3%, ‘\n바디케어\n’\n는\n4.1%\n줄었다\n.\n시장 규모가 커졌음에도 판매량이 감소한\xa0주요 원인으로\xa0화장품\xa0가격의\xa0증가가 꼽힌다.\n<연도별 러시아 화장품 판매수량\n및 증가율>\n(단위: 십\n억 개\n)\n[자료\n: Businesstat.ru]\n오프라인 판매는 감소하는 반면\xa0온라인\xa0마켓플레이스에서의\xa0판매가\xa0증가하고 있다\n. ’25\n년 온라인\xa0소매기업\n‘Wildberries’\n의 스킨케어 화장품 매출은\xa0전년 대비\n28%,\n색조 화장품 매출은\n20%\n증가했다\n.\n온라인 소매점들은 프리미엄\n,\n친환경\n,\n아시아 브랜드 등 다양한 카테고리의 메뉴를 운영하며 소비자들을 유입시키고 있다\n.\n제조\n·\n판매사 구분 모호\n,\n일반 소매점과\xa0뷰티 전문점간\xa0협력 확대\n물류\n‧\n포장 등 비용 상승은 기업이 제조와 판매를 함께\xa0통제하려는 수직통합 압력으로 작동했다\n.'),
 Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Pag

In [34]:
from pathlib import Path

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings.cache import CacheBackedEmbeddings

# 5. 임베딩 캐시 폴더 생성
Path("./data/embeddings").mkdir(parents=True, exist_ok=True)

embedder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

cache_dir = LocalFileStore("./data/embeddings/kotra_news_body_test")

cache_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedder,
    cache_dir,
    namespace="text-embedding-3-small"
)

In [35]:
# 6. FAISS 벡터스토어 생성
vectorstore = FAISS.from_documents(split_docs, cache_embedder)

# 7. Retriever 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [41]:
# 8. 검색 테스트
query = "러시아 화장품 시장 전망"

results = retriever.invoke(query)
results

[Document(id='160dba9e-5e49-47a0-a60a-c70bc1440b88', metadata={'row_id': 2, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240931', 'keywords': '#러시아, #식품, #외식, #식당, #호텔, #레스토랑, #Horeca, #전시회, #패스트푸드, #스트릿푸드, #커피, #식품가공기계, #주방, #푸드'}, page_content='앞서 언급된 바와 같이 외국산\n전문\xa0식품가공 및 주방\xa0설비에 대한 수요도 존재한다. 실제로 한국의 대러시아 식품가공기계 수출이 2024년 53.7%, 2025년 47.6%\xa0증가했으며, 식품포장기계도 2025년 수출액이 10.6% 늘었다. (MTI 4단위 기준)\n호텔 부문에서는 비용 절감과 서비스 표준화가 이뤄지며 디스펜서형\xa0어메니티 수요가 확대되고 있다. 동시에 고급 호텔을 중심으로 프리미엄 일회용 제품\xa0수요도 유지되고 있다. 또한,\n최근\n가족 단위 국내 관광이 늘면서 아동용 호텔 화장품이 틈새 품목으로 부상했다.\n전반적으로 러시아 HoReCa 시장은 품질을 유지하면서도 운영 효율성과 비용 절감을 중시하는 방향으로 변화하고 있다. 특히 소스류, 조미 원료, 호텔 소모품 등 B2B 분야에서 진출 기회가 열리고 있다. 부가가치가 높고 품질이 안정적이며, 수요에 맞게 공급 방식과 구성을 조정할 수 있는 제품의 시장성이 클 것으로 보인다.\n자료:\nHORECA EXPO\n공식 홈페이지, RBC, Kommersant, Forbes, METRO,\nKOTRA\n모스크바무역관 등'),
 Document(id='91636aeb-b06e-48c5-9134-82657e02c1ef', metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/li

# Qwen3-Embedding-0.6B-MLX-4bit 임베딩

In [20]:
# 1. CSV 읽고 첫 10개 본문 가져오기

import pandas as pd

df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

df = df[
    df["본문"].notna() &
    (df["본문"].astype(str).str.strip() != "")
].reset_index(drop=True)

df_sample = df.iloc[:10]
print(df_sample.shape)
df_sample.head()

/var/folders/wx/rj6wt5wn1yl8195jd9gqn5jc0000gn/T/ipykernel_3550/4108863917.py:5: DtypeWarning: Columns (0: 일자리구분명) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")


(10, 20)


,품목명한글,뉴스제목,게시물 URL,일자리구분명,HS코드명,저작권 정책,품목명영문,지역,파일링크,산업분류,산업분류코드목록,무역관정보,뉴스게시일자,게시글일련번호,뉴스작성자명,정보분류(메뉴ID),정보분류(메뉴명),국가,키워드,본문
0,"미용이나 메이크업용 제품류와 기초화장용 제품류[의약품은 제외하며, 선스크린(suns...",고물가와 경쟁 속 변화하는 러시아 뷰티 시장,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,3304,4유형,Beauty or make-up preparations and preparation...,유럽,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,노보시비르스크무역관,2026-05-04,240938,김하민,243,트렌드,러시아연방,"#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출",‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모...
1,NaN,"도미니카공화국, 플라스틱 제품류 수입, 유통 기업 대상 환경 규제 강화",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,중남미,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"화학,환경","I001212,I001203",산토도밍고무역관,2026-05-04,241022,Adriana Jimenez,242,경제·무역,도미니카공화국,"#도미니카공화국, #수입 규제, #고형 폐기물법, #환경 규제 준수, #플라스틱 규제",요약\n도미니카공화국은 2025년 12월 법률 제98-25호 시행을 통해 플라스틱·...
2,NaN,HORECA EXPO에서 살펴보는 러시아 외식산업 트렌드와 유망 분야,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,유럽,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,"농림수산식품,섬유/패션,생활소비재,유통/물류,관광/교육/서비스,산업일반","I001193,I001197,I001198,I001208,I001210,I001307",모스크바무역관,2026-05-04,240931,정의한,245,현장·인터뷰,러시아연방,"#러시아, #식품, #외식, #식당, #호텔, #레스토랑, #Horeca, #전시회...",<HORECA Expo 2026\n전시회\n개요\n>\n전시회명\nHORECA EX...
3,"미용이나 메이크업용 제품류와 기초화장용 제품류[의약품은 제외하며, 선스크린(suns...",[해외시장뉴스플러스] &lsquo;K-Lifestyle in Chennai&rsqu...,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,3304,4유형,Beauty or make-up preparations and preparation...,아시아,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,생활소비재,I001198,첸나이무역관,2026-05-04,240981,윤병은,243,트렌드,인도,"#뷰티, #화장품, #인도, #전자상거래, #온라인, #K-뷰티, #인플루언서, #인터뷰",인도\nK-\n뷰티 시장 확대\n인도 화장품 시장이 빠르게 커지고 있다. 시장조사 ...
4,NaN,2026 케냐 국제 투자 컨퍼런스를 통해 살펴본 케냐 투자유치 동향 및 전망,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,NaN,4유형,NaN,아프리카,https://dream.kotra.or.kr/ajaxa/fileCpnt/fileD...,산업일반,I001307,나이로비무역관,2026-05-04,241000,신정렬,322,투자진출,케냐,"#케냐, #투자, #컨퍼런스","최근 케냐는 '제조업 기반의 경제 성장'이라는 확고한 목표 아래, 파격적인 투자 환..."


In [21]:
# 2. LangChain Document 만들기
from langchain_core.documents import Document

docs = []

for idx, row in df_sample.iterrows():
    doc = Document(
        page_content=row["본문"],
        metadata={
            "row_id": idx,
            "url": row.get("게시물 URL", ""),
            "keywords": row.get("키워드", "")
        }
    )
    docs.append(doc)

print("문서 개수:", len(docs))
docs[0]

문서 개수: 10


Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938', 'keywords': '#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출'}, page_content='‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모는\n1\n조2000억\n루블\n(\n약\n144\n억 달러\n)\n로 전년 대비\n7%-8%\n증가했다\n. 반면\n판매되는 화장품\xa0수량은 감소세로\n, ‘Nilson’\n에 따르면 스킨케어 화장품 판매량은\n1\n년 만에\n13.3%, ‘\n헤어케어\n’\n는\n5.3%, ‘\n바디케어\n’\n는\n4.1%\n줄었다\n.\n시장 규모가 커졌음에도 판매량이 감소한\xa0주요 원인으로\xa0화장품\xa0가격의\xa0증가가 꼽힌다.\n<연도별 러시아 화장품 판매수량\n및 증가율>\n(단위: 십\n억 개\n)\n[자료\n: Businesstat.ru]\n오프라인 판매는 감소하는 반면\xa0온라인\xa0마켓플레이스에서의\xa0판매가\xa0증가하고 있다\n. ’25\n년 온라인\xa0소매기업\n‘Wildberries’\n의 스킨케어 화장품 매출은\xa0전년 대비\n28%,\n색조 화장품 매출은\n20%\n증가했다\n.\n온라인 소매점들은 프리미엄\n,\n친환경\n,\n아시아 브랜드 등 다양한 카테고리의 메뉴를 운영하며 소비자들을 유입시키고 있다\n.\n제조\n·\n판매사 구분 모호\n,\n일반 소매점과\xa0뷰티 전문점간\xa0협력 확대\n물류\n‧\n포장 등 비용 상승은 기업이 제조와 판매를 함께\xa0통제하려는 수직통합 압력으로 작동했다\n.\n소비자의\xa0구매력\xa0약화는\xa0멀티기능성\n,\n가성비\n,\n현지\xa0브랜드\xa0선호를\xa0키웠고\n,\n이는\xa0제조와\xa0판매\xa0모두에서\xa0더\xa0촘촘한\xa0가치사슬

In [57]:
# 3. 본문을 chunk로 나누기
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_docs = splitter.split_documents(docs)

print("원본 문서 수:", len(docs))
print("분할된 chunk 수:", len(split_docs))

split_docs

원본 문서 수: 10
분할된 chunk 수: 135


[Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938', 'keywords': '#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출'}, page_content='‘Alfabank’\n에 따르면\n’25\n년 러시아 화장품\n·\n향수 시장 규모는\n1\n조2000억\n루블\n(\n약\n144\n억 달러\n)\n로 전년 대비\n7%-8%\n증가했다\n. 반면\n판매되는 화장품\xa0수량은 감소세로\n, ‘Nilson’\n에 따르면 스킨케어 화장품 판매량은\n1\n년 만에\n13.3%, ‘\n헤어케어\n’\n는\n5.3%, ‘\n바디케어\n’\n는\n4.1%\n줄었다\n.\n시장 규모가 커졌음에도 판매량이 감소한\xa0주요 원인으로\xa0화장품\xa0가격의\xa0증가가 꼽힌다.\n<연도별 러시아 화장품 판매수량\n및 증가율>\n(단위: 십\n억 개\n)\n[자료\n: Businesstat.ru]\n오프라인 판매는 감소하는 반면\xa0온라인\xa0마켓플레이스에서의\xa0판매가\xa0증가하고 있다\n. ’25\n년 온라인\xa0소매기업\n‘Wildberries’\n의 스킨케어 화장품 매출은\xa0전년 대비\n28%,\n색조 화장품 매출은\n20%\n증가했다\n.\n온라인 소매점들은 프리미엄\n,\n친환경\n,\n아시아 브랜드 등 다양한 카테고리의 메뉴를 운영하며 소비자들을 유입시키고 있다\n.\n제조\n·\n판매사 구분 모호\n,\n일반 소매점과\xa0뷰티 전문점간\xa0협력 확대\n물류\n‧\n포장 등 비용 상승은 기업이 제조와 판매를 함께\xa0통제하려는 수직통합 압력으로 작동했다\n.'),
 Document(metadata={'row_id': 0, 'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Pag

In [23]:
# 4. Qwen3-Embedding MLX 4-bit 모델 로드
from mlx_embeddings import load
import mlx.core as mx
import numpy as np

model, tokenizer = load("majentik/Qwen3-Embedding-0.6B-MLX-4bit")

/Users/b58230717/Desktop/2026_2/aaa/project/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 132451.71it/s]


In [24]:
# 5. split_docs를 임베딩하기
doc_texts = [doc.page_content for doc in split_docs]

inputs = tokenizer(
    doc_texts,
    padding=True,
    truncation=True,
    max_length=8192,
    return_tensors="mlx"
)

outputs = model(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
)

doc_embeddings = np.array(outputs.text_embeds).astype("float32")

print("임베딩 shape:", doc_embeddings.shape)

NameError: name 'split_docs' is not defined

In [48]:
# 6. FAISS 인덱스 만들기
import faiss

dim = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print("FAISS에 저장된 chunk 수:", index.ntotal)

FAISS에 저장된 chunk 수: 135


In [18]:
# 7. 자연어 검색어 임베딩하기
query = "러시아 화장품 시장 전망"
task = "Given a Korean natural language search query, retrieve relevant KOTRA news passages."
query_text = f"Instruct: {task}\nQuery: {query}"

query_inputs = tokenizer(
    [query_text],
    padding=True,
    truncation=True,
    max_length=8192,
    return_tensors="mlx"
)

query_outputs = model(
    query_inputs["input_ids"],
    attention_mask=query_inputs["attention_mask"]
)

query_embedding = np.array(query_outputs.text_embeds).astype("float32")

print("query embedding shape:", query_embedding.shape)

NameError: name 'tokenizer' is not defined

In [55]:
# 8. 관련 chunk 검색하기
k = 50
scores, indices = index.search(query_embedding, k)

print("scores:", scores)
print("indices:", indices)

scores: [[0.7222228  0.7166976  0.6802507  0.6531437  0.638294   0.6031742
  0.59983474 0.57198143 0.5612768  0.5395051  0.5313067  0.521901
  0.5161246  0.4962517  0.48319215 0.46648148 0.44779748 0.4474364
  0.44293654 0.43524218 0.4186027  0.41467464 0.41008562 0.41000536
  0.40273744 0.38160598 0.37726542 0.37428907 0.3736775  0.36411145
  0.36041993 0.36041647 0.35973975 0.35939303 0.35821337 0.35689485
  0.35036105 0.34969148 0.34916392 0.34783107 0.3447735  0.34394708
  0.3413197  0.33289143 0.3314852  0.33021438 0.32837635 0.3283474
  0.32746425 0.32400206]]
indices: [[  0   2  14  12   4   9  13   7   6  10   5  96  91   3   8  97  92  90
   98 131 112 125  11  93 123  95 102 109 122   1 117 107 121 100  99 101
  132 133  79 120 105  94 118 128  82 130 113 114 103 106]]


In [56]:
# 9. 검색 결과 본문 그대로 출력
top_urls = []
seen_urls = set()

for score, idx in zip(scores[0], indices[0]):
    doc = split_docs[idx]
    url = doc.metadata.get("url", "")

    if url == "":
        continue

    if url not in seen_urls:
        seen_urls.add(url)
        top_urls.append({
            "url": url,
            "score": float(score),
            "row_id": doc.metadata.get("row_id"),
            "keywords": doc.metadata.get("keywords")
        })

    if len(top_urls) >= 5:
        break

top_urls

[{'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938',
  'score': 0.7222228050231934,
  'row_id': 0,
  'keywords': '#러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출'},
 {'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240772',
  'score': 0.52190101146698,
  'row_id': 7,
  'keywords': '#남성, #화장품, #뷰티, #멘즈뷰티, #K-Beauty, #일본, #일본 시장'},
 {'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240926',
  'score': 0.4474363923072815,
  'row_id': 6,
  'keywords': '#방재, #지진, #방범, #보안, #열사병'},
 {'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=241034',
  'score': 0.4352421760559082,
  'row_id': 9,
  'keywords': '#offshore, #해양에너지, #기자재, #미국, #해양, #장비, #공급, #운영지원, #MRO, #설비, #기기, #밸브, #피팅'},
 {'url': 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240791',
  'score': 0.41860270500183105,
  'row_id': 8,
  'keywords': '#푸드테크, #

# Qwen3-Embedding-0.6B-MLX-4bit 임베딩 (chunking x) 

In [25]:
import pandas as pd
from langchain_core.documents import Document
from langchain_core.documents import Document



df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

df = df[
    df["본문"].notna() &
    (df["본문"].astype(str).str.strip() != "")
].reset_index(drop=True)

df_sample = df.iloc[:10]

# 2. LangChain Document 만들기
docs = []

for idx, row in df_sample.iterrows():
    doc = Document(
        page_content=row["본문"],
        metadata={
            "row_id": idx,
            "url": row.get("게시물 URL", ""),
            "keywords": row.get("키워드", "")
        }
    )
    docs.append(doc)

doc_texts = [doc.page_content for doc in docs]

inputs = tokenizer(
    doc_texts,
    padding=True,
    truncation=True,
    max_length=32768,
    return_tensors="mlx"
)

outputs = model(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
)

doc_embeddings = np.array(outputs.text_embeds).astype("float32")
print("임베딩 shape:", doc_embeddings.shape)

/var/folders/wx/rj6wt5wn1yl8195jd9gqn5jc0000gn/T/ipykernel_3550/603946691.py:7: DtypeWarning: Columns (0: 일자리구분명) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/kotra_news_body.csv", encoding="utf-8-sig")


임베딩 shape: (10, 1024)


In [26]:
import faiss

dim = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print("FAISS 저장 문서 수:", index.ntotal)

FAISS 저장 문서 수: 10


In [27]:
query = "러시아 화장품 시장 전망"

task = "Given a Korean natural language search query, retrieve relevant KOTRA news articles."
query_text = f"Instruct: {task}\nQuery: {query}"

query_inputs = tokenizer(
    [query_text],
    padding=True,
    truncation=True,
    max_length=32768,
    return_tensors="mlx"
)

query_outputs = model(
    query_inputs["input_ids"],
    attention_mask=query_inputs["attention_mask"]
)

query_embedding = np.array(query_outputs.text_embeds).astype("float32")

scores, indices = index.search(query_embedding, 5)

for rank, idx in enumerate(indices[0], start=1):
    doc = docs[idx]
    score = scores[0][rank - 1]

    print("=" * 80)
    print(f"[검색 결과 {rank}]")
    print("score:", score)
    print("row_id:", doc.metadata.get("row_id"))
    print("url:", doc.metadata.get("url"))
    print("keywords:", doc.metadata.get("keywords"))

[검색 결과 1]
score: 0.7289356
row_id: 0
url: https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938
keywords: #러시아, #화장품, #뷰티, #뷰티트렌드, #화장품 수출
[검색 결과 2]
score: 0.5550666
row_id: 2
url: https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240931
keywords: #러시아, #식품, #외식, #식당, #호텔, #레스토랑, #Horeca, #전시회, #패스트푸드, #스트릿푸드, #커피, #식품가공기계, #주방, #푸드
[검색 결과 3]
score: 0.48456842
row_id: 7
url: https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240772
keywords: #남성, #화장품, #뷰티, #멘즈뷰티, #K-Beauty, #일본, #일본 시장
[검색 결과 4]
score: 0.41018942
row_id: 8
url: https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240791
keywords: #푸드테크, #DX, #AI, #식품, #자동화, #로봇
[검색 결과 5]
score: 0.35943317
row_id: 1
url: https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=241022
keywords: #도미니카공화국, #수입 규제, #고형 폐기물법, #환경 규제 준수, #플라스틱 규제


In [62]:
top_urls = []

for idx in indices[0]:
    doc = docs[idx]
    top_urls.append(doc.metadata.get("url"))

top_urls

['https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240938',
 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240931',
 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240772',
 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=240791',
 'https://dream.kotra.or.kr/user/extra/kotranews/bbs/linkView/jsp/Page.do?dataIdx=241022']

In [ ]:
# 32K 안에 들어오는지 확인
token_lengths = []

for text in df["본문"].iloc[:1000]:
    encoded = tokenizer(
        str(text),
        truncation=False,
        padding=False
    )
    token_lengths.append(len(encoded["input_ids"]))

pd.Series(token_lengths).describe()

count     1000.000000
mean      4552.194000
std       2384.714679
min         37.000000
25%       3064.250000
50%       4151.500000
75%       5825.500000
max      18081.000000
dtype: float64

## 전체 기사 단위 Qwen3 임베딩 저장

In [28]:
import gc
import json
from pathlib import Path

import faiss
import mlx.core as mx
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    model
    tokenizer
except NameError:
    from mlx_embeddings import load
    model, tokenizer = load("majentik/Qwen3-Embedding-0.6B-MLX-4bit")

CSV_PATH = Path("data/kotra_news_body.csv")
OUT_DIR = Path("data/faiss/kotra_news_body_qwen3_article")
BATCH_DIR = OUT_DIR / "batches"
INDEX_PATH = OUT_DIR / "index.faiss"
METADATA_PATH = OUT_DIR / "metadata.parquet"
CONFIG_PATH = OUT_DIR / "config.json"

MODEL_NAME = "majentik/Qwen3-Embedding-0.6B-MLX-4bit"
BATCH_SIZE = 1
MAX_LENGTH = 32768
CSV_ENCODING = "utf-8-sig"
TEXT_COL = "본문"

OUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_csv(CSV_PATH, encoding=CSV_ENCODING, dtype=str, keep_default_na=False)
df_all.columns = df_all.columns.str.replace("\ufeff", "", regex=False).str.strip()
df_all["source_row_id"] = df_all.index

df_embed = df_all[df_all[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
print("임베딩 대상 행 수:", len(df_embed))

metadata_cols = [
    "source_row_id",
    "뉴스제목",
    "게시물 URL",
    "키워드",
    "뉴스게시일자",
    "국가",
    "지역",
    "본문",
]
metadata_cols = [col for col in metadata_cols if col in df_embed.columns]
num_batches = (len(df_embed) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_id in tqdm(range(num_batches), desc="Qwen3 기사 임베딩"):
    emb_path = BATCH_DIR / f"embeddings_{batch_id:06d}.npy"
    meta_path = BATCH_DIR / f"metadata_{batch_id:06d}.parquet"

    if emb_path.exists() and meta_path.exists():
        continue

    start = batch_id * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(df_embed))
    batch_df = df_embed.iloc[start:end].copy()
    texts = batch_df[TEXT_COL].astype(str).tolist()

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="mlx"
    )
    outputs = model(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    embeddings = np.array(outputs.text_embeds).astype("float32")
    faiss.normalize_L2(embeddings)
    np.save(emb_path, embeddings)

    batch_meta = batch_df[metadata_cols].copy()
    batch_meta.insert(0, "faiss_id", range(start, end))
    batch_meta.to_parquet(meta_path, index=False)

    del inputs, outputs, embeddings, batch_df, batch_meta, texts
    gc.collect()
    mx.clear_cache()

batch_embedding_paths = sorted(BATCH_DIR.glob("embeddings_*.npy"))
batch_metadata_paths = sorted(BATCH_DIR.glob("metadata_*.parquet"))

if len(batch_embedding_paths) != len(batch_metadata_paths):
    raise ValueError("임베딩 batch와 metadata batch 개수가 다릅니다.")

index = None
for emb_path in tqdm(batch_embedding_paths, desc="FAISS 인덱스 생성"):
    embeddings = np.load(emb_path).astype("float32")
    if index is None:
        index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

metadata = pd.concat(
    [pd.read_parquet(path) for path in batch_metadata_paths],
    ignore_index=True
)
metadata["faiss_id"] = range(len(metadata))

faiss.write_index(index, str(INDEX_PATH))
metadata.to_parquet(METADATA_PATH, index=False)

config = {
    "model_name": MODEL_NAME,
    "source_csv": str(CSV_PATH),
    "index_path": str(INDEX_PATH),
    "metadata_path": str(METADATA_PATH),
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "chunking": False,
    "normalized_l2": True,
    "rows": int(len(metadata)),
    "embedding_dim": int(index.d),
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

print("FAISS index:", INDEX_PATH)
print("metadata:", METADATA_PATH)
print("rows:", index.ntotal)
print("dim:", index.d)

임베딩 대상 행 수: 94550


Qwen3 기사 임베딩:   3%|▎         | 2386/94550 [02:35<1:40:07, 15.34it/s] 


KeyboardInterrupt: 

In [4]:
import faiss
import numpy as np
import pandas as pd

index = faiss.read_index(str(INDEX_PATH))
metadata = pd.read_parquet(METADATA_PATH)

query = "러시아 화장품 시장 전망"
task = "Given a Korean natural language search query, retrieve relevant KOTRA news articles."
query_text = f"Instruct: {task}\nQuery: {query}"

query_inputs = tokenizer(
    [query_text],
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="mlx"
)
query_outputs = model(
    query_inputs["input_ids"],
    attention_mask=query_inputs["attention_mask"]
)

query_embedding = np.array(query_outputs.text_embeds).astype("float32")
faiss.normalize_L2(query_embedding)

scores, indices = index.search(query_embedding, 10)

results = metadata.iloc[indices[0]].copy()
results.insert(0, "score", scores[0])
results[["score", "뉴스제목", "게시물 URL", "키워드", "뉴스게시일자", "국가"]].head(10)

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char *) at /Users/runner/work/faiss-wheels/faiss-wheels/third-party/faiss/faiss/impl/io.cpp:70: Error: 'f' failed: could not open data/faiss/kotra_news_body_qwen3_article/index.faiss for reading: No such file or directory

# LLM 요약